In [4]:
import re
import json

JS_FILE = "speciesIndexcopy.js"

with open(JS_FILE, "r", encoding="utf-8") as f:
    content = f.read()

# -----------------------------
# 1️⃣ 提取 speciesList 数组
# -----------------------------
match = re.search(
    r'export\s+const\s+speciesList\s*=\s*(\[[\s\S]*?\])\s*;',
    content
)

if not match:
    raise ValueError("❌ speciesList not found")

species_list = json.loads(match.group(1))

print(f"✔ Extracted {len(species_list)} species")

# -----------------------------
# 2️⃣ 生成 name -> id
# -----------------------------
species_index = {
    item["name"]: item["id"]
    for item in species_list
}

# -----------------------------
# 3️⃣ 生成 id -> name
# -----------------------------
reverse_species_index = {
    str(item["id"]): item["name"]   # JSON key 必须是字符串
    for item in species_list
}

# -----------------------------
# 4️⃣ 保存 JSON 文件
# -----------------------------
with open("speciesIndexcopy.json", "w", encoding="utf-8") as f:
    json.dump(species_index, f, indent=2)

with open("reverseSpeciesIndexcopy.json", "w", encoding="utf-8") as f:
    json.dump(reverse_species_index, f, indent=2)

print("✔ Created speciesIndex.json")
print("✔ Created reverseSpeciesIndex.json")

✔ Extracted 277 species
✔ Created speciesIndex.json
✔ Created reverseSpeciesIndex.json


In [6]:
import re
import json

INPUT_JS = "speciesAliascopy.js"
OUTPUT_JSON = "speciesAliascopy.json"

with open(INPUT_JS, "r", encoding="utf-8") as f:
    content = f.read()

# 提取 export const speciesAlias = {...};
match = re.search(
    r'export\s+const\s+speciesAlias\s*=\s*({[\s\S]*?})\s*;',
    content
)

if not match:
    raise ValueError("❌ speciesAlias not found in JS file")

alias_dict = json.loads(match.group(1))

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(alias_dict, f, indent=2)

print(f"✔ Created {OUTPUT_JSON}")
print("Total aliases:", len(alias_dict))

✔ Created speciesAliascopy.json
Total aliases: 8


In [8]:
import os
import json

# ============================
# 配置
# ============================

GRID_BASE_DIR = "grid"
ZOOM_FOLDERS = ["zoom8", "zoom10", "zoom12", "zoom14"]

SPECIES_INDEX_FILE = "speciesIndexcopy.json"
SPECIES_ALIAS_FILE = "speciesAliascopy.json"


# ============================
# 读取 species index
# ============================

with open(SPECIES_INDEX_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 如果是 name->id 的 dict
if isinstance(raw_data, dict):
    species_name_to_id = raw_data

# 如果是 list 结构
elif isinstance(raw_data, list):
    species_name_to_id = {s["name"]: s["id"] for s in raw_data}

else:
    raise ValueError("Unsupported speciesIndex structure")

# ============================
# 读取 alias（可选）
# ============================

if os.path.exists(SPECIES_ALIAS_FILE):
    with open(SPECIES_ALIAS_FILE, "r", encoding="utf-8") as f:
        species_alias = json.load(f)
else:
    species_alias = {}

print("Species loaded:", len(species_name_to_id))
print("Alias loaded:", len(species_alias))


# ============================
# 工具函数
# ============================

def resolve_name_to_id(name):
    """
    name -> canonical name -> id
    """
    if name in species_name_to_id:
        return species_name_to_id[name]

    # alias 映射
    if name in species_alias:
        canonical = species_alias[name]
        return species_name_to_id.get(canonical)

    return None


def compute_centroid(feature):
    """
    计算 polygon 中心（用于 heatmap）
    """
    coords = feature.get("geometry", {}).get("coordinates", [])
    if not coords or not isinstance(coords[0], list):
        return None

    ring = coords[0]

    lons = [pt[0] for pt in ring]
    lats = [pt[1] for pt in ring]

    if not lons or not lats:
        return None

    return [
        (min(lons) + max(lons)) / 2,
        (min(lats) + max(lats)) / 2
    ]


# ============================
# 主处理循环
# ============================

for zoom in ZOOM_FOLDERS:

    zoom_path = os.path.join(GRID_BASE_DIR, zoom)

    if not os.path.exists(zoom_path):
        continue

    print(f"\nProcessing {zoom}")

    for filename in os.listdir(zoom_path):

        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(zoom_path, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        features = data.get("features", [])

        for feature in features:

            props = feature.get("properties", {})

            # ============================
            # 1️⃣ speciesTop name -> id
            # ============================

            species_top = props.get("speciesTop", [])

            new_species_top = []

            for item in species_top:

                if not isinstance(item, list) or len(item) != 2:
                    continue

                name_or_id, count = item

                # 已经是 id（数字）
                if isinstance(name_or_id, int):
                    new_species_top.append([name_or_id, count])
                    continue

                # 如果是字符串
                if isinstance(name_or_id, str):
                    sid = resolve_name_to_id(name_or_id)

                    if sid is not None:
                        new_species_top.append([sid, count])
                    else:
                        print(f"⚠ Unmapped species: {name_or_id}")
                        continue

            props["speciesTop"] = new_species_top

            # ============================
            # 2️⃣ 加 feature.id（如果没有）
            # ============================

            if "id" not in feature:
                feature["id"] = None  # 先占位

            # ============================
            # 3️⃣ 加 centroid
            # ============================

            centroid = compute_centroid(feature)
            if centroid:
                props["centroid"] = centroid

        # 给 feature.id 统一编号
        for i, feature in enumerate(features):
            feature["id"] = i

        # 保存
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, separators=(",", ":"))

        print(f"✔ Updated {filename} | features: {len(features)}")

print("\nAll grid files updated successfully.")

Species loaded: 277
Alias loaded: 8

Processing zoom8
✔ Updated 6.json | features: 690
✔ Updated 7.json | features: 641
✔ Updated 10.json | features: 638
✔ Updated 1.json | features: 717
✔ Updated 11.json | features: 587
✔ Updated 2.json | features: 690
✔ Updated 12.json | features: 639
✔ Updated 3.json | features: 696
✔ Updated 8.json | features: 614
✔ Updated 4.json | features: 726
✔ Updated 5.json | features: 728
✔ Updated all.json | features: 1025
✔ Updated 9.json | features: 621

Processing zoom10
✔ Updated 6.json | features: 1304
✔ Updated 7.json | features: 1179
✔ Updated 10.json | features: 1160
✔ Updated 1.json | features: 1325
✔ Updated 11.json | features: 1022
✔ Updated 2.json | features: 1244
✔ Updated 12.json | features: 1142
✔ Updated 3.json | features: 1320
✔ Updated 8.json | features: 1088
✔ Updated 4.json | features: 1382
✔ Updated 5.json | features: 1428
✔ Updated all.json | features: 2909
✔ Updated 9.json | features: 1105

Processing zoom12
✔ Updated 6.json | feature

In [11]:
pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import json
import numpy as np
from sklearn.neighbors import KernelDensity


# ============================
# 配置
# ============================

GRID_BASE_DIR = "grid"
ZOOM_FOLDERS = ["zoom8", "zoom10", "zoom12", "zoom14"]

# 每个 zoom 对应不同 bandwidth（单位：经纬度度数）
# ⚠ 可调
BANDWIDTH_BY_ZOOM = {
    "zoom8": 0.08,
    "zoom10": 0.04,
    "zoom12": 0.02,
    "zoom14": 0.01
}


# ============================
# KDE 计算函数
# ============================

def compute_density(features, bandwidth):
    """
    使用 weighted KDE 计算每个 feature 的 density
    """

    points = []
    weights = []

    for f in features:
        props = f.get("properties", {})
        centroid = props.get("centroid")
        total = props.get("totalCount", 0)

        if centroid and total > 0:
            points.append(centroid)
            weights.append(total)

    if len(points) == 0:
        return

    points = np.array(points)
    weights = np.array(weights)

    # 构建 KDE
    kde = KernelDensity(
        bandwidth=bandwidth,
        kernel="gaussian"
    )

    kde.fit(points, sample_weight=weights)

    # 在相同点上计算 density
    log_density = kde.score_samples(points)
    density = np.exp(log_density)

    # 归一化（避免数值太大）
    density = density / density.max()

    # 写回
    idx = 0
    for f in features:
        props = f.get("properties", {})
        centroid = props.get("centroid")
        total = props.get("totalCount", 0)

        if centroid and total > 0:
            props["density"] = float(density[idx])
            idx += 1
        else:
            props["density"] = 0.0


# ============================
# 主循环
# ============================

for zoom in ZOOM_FOLDERS:

    zoom_path = os.path.join(GRID_BASE_DIR, zoom)

    if not os.path.exists(zoom_path):
        continue

    bandwidth = BANDWIDTH_BY_ZOOM.get(zoom, 0.02)

    print(f"\nProcessing {zoom} | bandwidth={bandwidth}")

    for filename in os.listdir(zoom_path):

        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(zoom_path, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        features = data.get("features", [])

        if not features:
            continue

        print(f"  ➜ {filename} | features: {len(features)}")

        compute_density(features, bandwidth)

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, separators=(",", ":"))

        print(f"    ✔ density written")

print("\nAll grid files updated with density.")


Processing zoom8 | bandwidth=0.08
  ➜ 6.json | features: 690
    ✔ density written
  ➜ 7.json | features: 641
    ✔ density written
  ➜ 10.json | features: 638
    ✔ density written
  ➜ 1.json | features: 717
    ✔ density written
  ➜ 11.json | features: 587
    ✔ density written
  ➜ 2.json | features: 690
    ✔ density written
  ➜ 12.json | features: 639
    ✔ density written
  ➜ 3.json | features: 696
    ✔ density written
  ➜ 8.json | features: 614
    ✔ density written
  ➜ 4.json | features: 726
    ✔ density written
  ➜ 5.json | features: 728
    ✔ density written
  ➜ all.json | features: 1025
    ✔ density written
  ➜ 9.json | features: 621
    ✔ density written

Processing zoom10 | bandwidth=0.04
  ➜ 6.json | features: 1304
    ✔ density written
  ➜ 7.json | features: 1179
    ✔ density written
  ➜ 10.json | features: 1160
    ✔ density written
  ➜ 1.json | features: 1325
    ✔ density written
  ➜ 11.json | features: 1022
    ✔ density written
  ➜ 2.json | features: 1244
    ✔ 